# 1. Set up

## 1.1 Kernel

- For training users, please select the kernel "training"
- After choosing your kernel, wait for a few minutes until the kernel initializes properly

## 1.2 Spark Session
A spark session named ```spark``` is already built for you based on the configuration of your chosen template

The AppName is the same name in the Ocean Spark Dashboard. It is good practice to monitor your notebook in the dashboard. The notebook status should be "Running" (blue circle)

# 2. Import packages

In [ ]:
import os
import pandas as pd
from datetime import datetime

In [ ]:
from ais import functions as af

In [ ]:
import pyspark.sql.functions as F

In [ ]:
import h3
import folium
from shapely.geometry import Polygon

/opt/spark/python/lib/pyspark.zip/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.


Closing down clientserver connection


# 3. Read AIS Data

AIS data and ship register data are stored in Amazon S3 buckets.
We prepared a sample dataset for training users, which is a subset of the AIS data.
The training dataset contains AIS records of January 2024.

There are two ways to read the AIS data.
- Set a basepath, and read the AIS data using spark.read()
- Use the ais package to read the AIS data

## 3.1 Use spark.read() 

- Get the base path

In [ ]:
#get the base path
basepath = os.environ["AIS_S3PATH"]
basepath

's3a://ungp-cherryl-582958291898-hjnno/exact-earth-data/transformed/prod/'

- Read ALL AIS for one date 

Read from path ```basepath + "year=<YYYY>/month=<mm>/day=<dd>"```

In [ ]:
df1 = spark.read.option("basepath",basepath).parquet(basepath+"year=2024/month=01/day=01")

In [ ]:
df1.limit(1).show(vertical=True, truncate=True)

-RECORD 0---------------------------------
 message_type      | 1                    
 mmsi              | 241575000            
 dt_insert_utc     | 2024-01-01 15:45:43  
 longitude         | 12.245665            
 latitude          | 44.44254833          
 imo               | 9405796              
 vessel_name       | SEAVEN VOYAGER       
 callsign          | SVCS4                
 vessel_type       | Tanker               
 vessel_type_code  | 80                   
 vessel_type_cargo | NULL                 
 vessel_class      | A                    
 length            | 100.0                
 width             | 18.0                 
 flag_country      | Greece               
 flag_code         | 241                  
 destination       | EG DAM > IT RAN      
 eta               | 12310200             
 draught           | 6.5                  
 sog               | 0.0                  
 cog               | 153.0                
 rot               | 0.0                  
 heading   

- Read AIS data of a list of dates

In [ ]:
#Read for one-week
#puth all date paths in a list
df_week = spark.read.parquet(*[basepath + "year=2024/month=01/day=01",
                          basepath + "year=2024/month=01/day=02",
                          basepath + "year=2024/month=01/day=03",
                          basepath + "year=2024/month=01/day=04",
                          basepath + "year=2024/month=01/day=05",
                          basepath + "year=2024/month=01/day=06",
                          basepath + "year=2024/month=01/day=07"]
                       )

In [ ]:
#Create a date column and then count rows per date
df_week = df_week.withColumn('date', F.to_date("dt_insert_utc"))
df_week.groupBy('date').count().show()

+----------+-------+
|      date|  count|
+----------+-------+
|2024-01-02|3401589|
|2024-01-01|3375806|
|2024-01-03|3381960|
|2024-01-04|3393634|
|2024-01-05|3398235|
|2024-01-06|3397700|
|2024-01-07|3378189|
|2024-01-08|   5830|
+----------+-------+



## 3.2 Use ais package

- Read AIS data of a certain day

In [ ]:
start_date = datetime.fromisoformat("2024-01-01")
df2 = af.get_ais(spark,start_date)

In [ ]:
df2.limit(1).show(vertical=True, truncate=True)

-RECORD 0---------------------------------
 message_type      | 1                    
 mmsi              | 241575000            
 dt_insert_utc     | 2024-01-01 15:45:43  
 longitude         | 12.245665            
 latitude          | 44.44254833          
 imo               | 9405796              
 vessel_name       | SEAVEN VOYAGER       
 callsign          | SVCS4                
 vessel_type       | Tanker               
 vessel_type_code  | 80                   
 vessel_type_cargo | NULL                 
 vessel_class      | A                    
 length            | 100.0                
 width             | 18.0                 
 flag_country      | Greece               
 flag_code         | 241                  
 destination       | EG DAM > IT RAN      
 eta               | 12310200             
 draught           | 6.5                  
 sog               | 0.0                  
 cog               | 153.0                
 rot               | 0.0                  
 heading   

- read AIS data of a date range

NOTE: For complex calculations, it is recommended to read only maximum of 1 month of data.

In [ ]:
start_date = datetime.fromisoformat("2024-01-01")
end_date = datetime.fromisoformat("2024-01-05")
df3 = af.get_ais(spark,start_date, end_date = end_date)

In [ ]:
df3.withColumn("date",F.col("dt_insert_utc").cast("date")) \
        .groupby('date')  \
        .agg(F.count("eeid").alias("count")) \
        .orderBy("date") \
.show()

+----------+-------+
|      date|  count|
+----------+-------+
|2024-01-01|3375806|
|2024-01-02|3401589|
|2024-01-03|3381960|
|2024-01-04|3393634|
|2024-01-05|3398235|
|2024-01-06|   3007|
+----------+-------+



- read AIS data of a list of dates

In [ ]:
date_list = [datetime.fromisoformat("2024-01-01"),
             datetime.fromisoformat("2024-01-08"),
             datetime.fromisoformat("2024-01-15")]
date_list

[datetime.datetime(2024, 1, 1, 0, 0),
 datetime.datetime(2024, 1, 8, 0, 0),
 datetime.datetime(2024, 1, 15, 0, 0)]

In [ ]:
df4 = af.get_ais(spark,date_list=date_list)

In [ ]:
df4.withColumn("date",F.col("dt_insert_utc").cast("date")) \
        .groupby('date')  \
        .agg(F.count("eeid").alias("count")) \
        .orderBy("date") \
.show()

+----------+-------+
|      date|  count|
+----------+-------+
|2024-01-01|3375806|
|2024-01-02|   3719|
|2024-01-08|3407798|
|2024-01-09|   4390|
|2024-01-15|3424735|
|2024-01-16|   8107|
+----------+-------+



# 4. How to get AIS data of a certain place

## 4.1 Get H3 Index

We can get the H3 index of any pair of latitude and longitude using geo_to_h3 function from h3 package.

In [ ]:
#Eg:point in Suez Canal
latitude = 30.56529681740963
longitude = 32.33832930701776

In [ ]:
#Resolution of H3 Index used in partitioning
h3index_0 = h3.geo_to_h3(latitude, longitude, 0) #resolution 0
print(h3index_0)
h3index_1 = h3.geo_to_h3(latitude, longitude, 1) #resolution 1
print(h3index_1)
h3index_2 = h3.geo_to_h3(latitude, longitude, 2) #resolution 2
print(h3index_2)

803ffffffffffff
813e7ffffffffff
823e67fffffffff


 - We can visualize this using **folium** package. From the map, we see that Suez Canal is contained within H3 '803ffffffffffff'

In [ ]:
m = folium.Map(location=[latitude, longitude], zoom_start=4, tiles="cartodbpositron")
folium.Marker([latitude, longitude]).add_to(m)
folium.GeoJson(Polygon(h3.h3_to_geo_boundary(h3index_0, geo_json=True))).add_to(m)
folium.GeoJson(Polygon(h3.h3_to_geo_boundary(h3index_1, geo_json=True))).add_to(m)
folium.GeoJson(Polygon(h3.h3_to_geo_boundary(h3index_2, geo_json=True))).add_to(m)
m

# 4.2 filter data

Get the AIS records in side the H3 geo boundary

 - Resolution 0: 803ffffffffffff

In [ ]:
#If we want to read all the records within this area. 
#We can use the spark.sql(), and cast the H3 index.
df1.createOrReplaceTempView("temp_df")
df_H3_0 = spark.sql("""
                select dt_pos_utc, mmsi, longitude, latitude, H3index_0, H3_int_index_0, H3_int_index_1, H3_int_index_2, H3_int_index_3
                from temp_df
                where H3index_0 = "803ffffffffffff"
                """)

In [ ]:
df_H3_0.count()

60444

- Resolution 1: 813e7ffffffffff

In [ ]:
h3index_1_int = int(h3index_1,16)
h3index_1_int

582063863558569983

In [ ]:
df1.createOrReplaceTempView("temp_df")
df_H3_1 = spark.sql("""
                select dt_pos_utc, mmsi, longitude, latitude, H3index_0, H3_int_index_0, H3_int_index_1, H3_int_index_2, H3_int_index_3
                from temp_df
                where H3_int_index_1 = 582063863558569983
                """)

In [ ]:
df_H3_1.count()

7989

# 5. Save and Download Data

## 5.1 Transform to DataFrame

In [ ]:
dataframe_H3_1 = df_H3_1.toPandas()
dataframe_H3_1.head(10)

,dt_pos_utc,mmsi,longitude,latitude,H3index_0,H3_int_index_0,H3_int_index_1,H3_int_index_2,H3_int_index_3
0,2024-01-01 12:31:35,438031041,32.971667,28.530000,803ffffffffffff,577586652210266111,582063863558569983,586563614895243263,591067077083660287
1,2024-01-01 08:43:54,341206001,32.581303,30.008583,803ffffffffffff,577586652210266111,582063863558569983,586565813918498815,591069138667962367
2,2024-01-01 23:59:41,538009926,33.326667,28.130000,803ffffffffffff,577586652210266111,582063863558569983,586563614895243263,591067077083660287
3,2024-01-01 18:33:01,622123249,32.586012,29.962367,803ffffffffffff,577586652210266111,582063863558569983,586565813918498815,591069138667962367
4,2024-01-01 15:38:30,622166811,32.571912,29.942052,803ffffffffffff,577586652210266111,582063863558569983,586565813918498815,591069138667962367
5,2024-01-01 01:27:22,311001298,32.368333,29.598333,803ffffffffffff,577586652210266111,582063863558569983,586565813918498815,591069138667962367
6,2024-01-01 03:25:28,356666000,32.328893,31.195647,803ffffffffffff,577586652210266111,582063863558569983,586565813918498815,591069276106915839
7,2024-01-01 13:09:04,538009926,32.533117,30.250025,803ffffffffffff,577586652210266111,582063863558569983,586565813918498815,591068932509532159
8,2024-01-01 00:00:27,275529000,31.671667,31.646667,803ffffffffffff,577586652210266111,582063863558569983,586565813918498815,591069207387439103
9,2024-01-01 10:37:22,525008067,32.636667,29.716667,803ffffffffffff,577586652210266111,582063863558569983,586565813918498815,591069138667962367


## 5.2 Download Data

You can use create_download_link() function to download data. But please note that the load is limited. Do not try to download large dataset.

In [ ]:
af.create_download_link(dataframe_H3_1.head(100), title = "Download CSV file", filename = "SampleResult.csv")

# 6.  Map for AIS data

In [ ]:
import folium
from folium.plugins import HeatMap
from shapely.geometry import Polygon, MultiPoint, mapping
import h3

In [ ]:
def create_geojson (h3_list):
    geojson = {"type": 'FeatureCollection',
               "features": [{ "type": "Feature",
                            "geometry": {"type":"Polygon",
                                         "coordinates": (h3.h3_to_geo_boundary(x,geo_json=True),)
                                        },
                            "properties": {"H3 index":x,"Resolution":h3.h3_get_resolution(x)}, 
                           } for x in h3_list]
               }
    return geojson

In [ ]:
#In this example we choose the records near Suez Canal on Jan 1st 2024 as an example
#transform to Pandas format
df = df_H3_0.toPandas()
df

,dt_pos_utc,mmsi,longitude,latitude,H3index_0,H3_int_index_0,H3_int_index_1,H3_int_index_2,H3_int_index_3
0,2024-01-01 02:36:49,241656000,23.704383,37.915330,803ffffffffffff,577586652210266111,581509709698170879,586011110302285823,590514366332272639
1,2024-01-01 03:26:25,477858500,23.086392,37.915357,803ffffffffffff,577586652210266111,581509709698170879,586579557813846015,590514366332272639
2,2024-01-01 13:31:36,237338400,23.636667,35.516667,803ffffffffffff,577586652210266111,582081455744614399,586584505616171007,591087761646157823
3,2024-01-01 07:56:45,256796000,14.505492,35.887648,803ffffffffffff,577586652210266111,582077057698103295,586580107569659903,591083226160693247
4,2024-01-01 01:30:55,256579000,23.588953,37.954755,803ffffffffffff,577586652210266111,581509709698170879,586011110302285823,590514366332272639
...,...,...,...,...,...,...,...,...,...
60439,2024-01-01 23:55:50,229056000,23.668278,37.938227,803ffffffffffff,577586652210266111,581509709698170879,586011110302285823,590514366332272639
60440,2024-01-01 09:47:50,210989000,23.555750,37.902167,803ffffffffffff,577586652210266111,581509709698170879,586011110302285823,590514366332272639
60441,2024-01-01 09:28:23,622123255,32.310860,31.256988,803ffffffffffff,577586652210266111,582063863558569983,586565813918498815,591069276106915839
60442,2024-01-01 08:20:57,256002123,23.648098,37.931028,803ffffffffffff,577586652210266111,581509709698170879,586011110302285823,590514366332272639


# 6.1 Heat map

In [ ]:
m = folium.Map(location=[latitude, longitude], zoom_start=6, tiles="cartodbpositron")
a = HeatMap(df[['latitude','longitude']], radius=10, blur=10, name="AIS Data HeatMap").add_to(m)
m

# 6.2 Map data with H3 resolutions

In [ ]:
df['h3_1'] = df[['latitude','longitude']].apply(lambda x: h3.geo_to_h3(x[0],x[1],1), axis=1)
df['h3_2'] = df[['latitude','longitude']].apply(lambda x: h3.geo_to_h3(x[0],x[1],2), axis=1)
df['h3_3'] = df[['latitude','longitude']].apply(lambda x: h3.geo_to_h3(x[0],x[1],3), axis=1)

In [ ]:
a = folium.GeoJson(create_geojson(df["h3_1"].unique().tolist()),
                    tooltip = folium.GeoJsonTooltip(fields=['H3 index','Resolution']),
                    name='H3 Resolution 1', show=False).add_to(m)
a = folium.GeoJson(create_geojson(df["h3_2"].unique().tolist()),
                    tooltip = folium.GeoJsonTooltip(fields=['H3 index','Resolution']),
                    name='H3 Resolution 2', show=False).add_to(m)
a = folium.GeoJson(create_geojson(df["h3_3"].unique().tolist()),
                    tooltip = folium.GeoJsonTooltip(fields=['H3 index','Resolution']),
                    name='H3 Resolution 3', show=False).add_to(m)
folium.LayerControl().add_to(m)
m

# 5. Stop Spark Session

Always end the spark session to free any executors assigned to your tasks, and shut down the kernel.

In [ ]:
spark.stop()